# SQL query from table names - Continued

In [1]:
import os
from google.colab import userdata
from openai import OpenAI

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise ValueError("OPENAI_API_KEY was not found in Colab Secrets.")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

print("OPENAI_API_KEY loaded successfully into the environment.")

OPENAI_API_KEY loaded successfully into the environment.


## The old Prompt

In [3]:
#The old prompt
old_context = [ {'role':'system', 'content':"""
you are a bot to assist in create SQL commands, all your answers should start with \
this is your SQL, and after that an SQL that can do what the user request. \
Your Database is composed by a SQL database with some tables. \
Try to maintain the SQL order simple.
Put the SQL command in white letters with a black background, and just after \
a simple and concise text explaining how it works.
If the user ask for something that can not be solved with an SQL Order \
just answer something nice and simple, maximum 10 words, asking him for something that \
can be solved with SQL.
"""} ]

old_context.append( {'role':'system', 'content':"""
first table:
{
  "tableName": "employees",
  "fields": [
    {
      "nombre": "ID_usr",
      "tipo": "int"
    },
    {
      "nombre": "name",
      "tipo": "varchar"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
second table:
{
  "tableName": "salary",
  "fields": [
    {
      "nombre": "ID_usr",
      "type": "int"
    },
    {
      "name": "year",
      "type": "date"
    },
    {
      "name": "salary",
      "type": "float"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
third table:
{
  "tablename": "studies",
  "fields": [
    {
      "name": "ID",
      "type": "int"
    },
    {
      "name": "ID_usr",
      "type": "int"
    },
    {
      "name": "educational_level",
      "type": "int"
    },
    {
      "name": "Institution",
      "type": "varchar"
    },
    {
      "name": "Years",
      "type": "date"
    }
    {
      "name": "Speciality",
      "type": "varchar"
    }
  ]
}
"""
})

## New Prompt.
We are going to improve it following the instructions of a Paper from the Ohaio University: [How to Prompt LLMs for Text-to-SQL: A Study in Zero-shot, Single-domain, and Cross-domain Settings](https://arxiv.org/abs/2305.11853). I recommend you read that paper.

For each table, we will define the structure using the same syntax as in a SQL create table command, and add the sample rows of the content.

Finally, at the end of the prompt, we'll include some example queries with the SQL that the model should generate. This technique is called Few-Shot Samples, in which we provide the prompt with some examples to assist it in generating the correct SQL.


In [8]:
context = [ {'role':'system', 'content':"""

CREATE TABLE employees (
    ID_Usr INTEGER PRIMARY KEY,
    name VARCHAR(100)
);

CREATE TABLE salary (
    ID_Usr INTEGER,
    year DATE,
    salary FLOAT,
    FOREIGN KEY (ID_Usr) REFERENCES employees(ID_Usr)
);

CREATE TABLE studies (
    ID INTEGER PRIMARY KEY,
    ID_Usr INTEGER,
    educational_level INTEGER,
    Institution VARCHAR(100),
    Years DATE,
    Speciality VARCHAR(100),
    FOREIGN KEY (ID_Usr) REFERENCES employees(ID_Usr)
);
"""
}]


In [9]:
#FEW SHOT SAMPLES
context.append( {'role':'system', 'content':"""
 -- Maintain the SQL order simple and efficient as you can, using valid SQL Lite, answer the following questions for the table provided above.

Question: Which department has the highest average salary?

SELECT d.department_name, AVG(s.salary) AS average_salary
FROM departments d
JOIN employees e ON d.department_id = e.department_id
JOIN salaries s ON e.employee_id = s.employee_id
GROUP BY d.department_name
ORDER BY average_salary DESC
LIMIT 1;

Question: How many employees are in each department?

SELECT d.department_name, COUNT(e.employee_id) AS number_of_employees
FROM departments d
LEFT JOIN employees e ON d.department_id = e.department_id
GROUP BY d.department_name;

Question: Which employees have the highest salary?

SELECT e.employee_name, s.salary
FROM employees e
JOIN salaries s ON e.employee_id = s.employee_id
ORDER BY s.salary DESC
LIMIT 5;

Question: What is the average salary by education level?

SELECT ed.education_level, AVG(s.salary) AS average_salary
FROM education ed
JOIN employees e ON ed.employee_id = e.employee_id
JOIN salaries s ON e.employee_id = s.employee_id
GROUP BY ed.education_level;
"""
})

In [10]:
#Functio to call the model.
def return_CCRMSQL(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=0,
        )

    return (response.choices[0].message.content)

## NL2SQL Samples
We're going to review some examples generated with the old prompt and others with the new prompt.

In [12]:
#new
context_user = context.copy()
print(return_CCRMSQL("Which department has the highest average salary?", context_user))

SELECT d.department_name, AVG(s.salary) AS average_salary
FROM departments d
JOIN employees e ON d.department_id = e.department_id
JOIN salaries s ON e.employee_id = s.employee_id
GROUP BY d.department_name
ORDER BY average_salary DESC
LIMIT 1;


In [13]:
#old
old_context_user = old_context.copy()
print(return_CCRMSQL("Which department has the highest average salary?", old_context_user))

This is your SQL:
```sql
SELECT department, AVG(salary) AS avg_salary
FROM employees
JOIN salary ON employees.ID_usr = salary.ID_usr
GROUP BY department
ORDER BY avg_salary DESC
LIMIT 1;
```

This SQL query joins the "employees" and "salary" tables on the employee ID, calculates the average salary for each department, orders the results by average salary in descending order, and then selects the department with the highest average salary.


In [14]:
#new
print(return_CCRMSQL("What is the average salary by education level?", context_user))

SELECT educational_level, AVG(salary) AS average_salary
FROM studies
JOIN salary ON studies.ID_Usr = salary.ID_Usr
GROUP BY educational_level;


In [15]:
#old
print(return_CCRMSQL("What is the average salary by education level?", old_context_user))

This is your SQL:
```sql
SELECT s.educational_level, AVG(s.salary) AS average_salary
FROM salary s
JOIN studies st ON s.ID_usr = st.ID_usr
GROUP BY s.educational_level;
```

This SQL query retrieves the average salary for each education level by joining the "salary" table with the "studies" table on the user ID. It then calculates the average salary for each education level group using the AVG function and groups the results by educational level.


# Exercise
 - Complete the prompts similar to what we did in class.
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong.
     - What did you learn?

In [16]:
exercise_questions = [
    "Which employees have the highest salary within each education level?",
    "How many employees graduated from each university, and what is the average salary for each university?",
    "Which schools produced the highest paid employees?",
    "What is the total company revenue by department?",
    "What is the average education level of employees?"
]

print("NEW PROMPT RESULTS")
for q in exercise_questions:
    print("QUESTION:", q)
    print(return_CCRMSQL(q, context_user))
    print("-" * 80)

print("OLD PROMPT RESULTS")
for q in exercise_questions:
    print("QUESTION:", q)
    print(return_CCRMSQL(q, old_context_user))
    print("-" * 80)

NEW PROMPT RESULTS
QUESTION: Which employees have the highest salary within each education level?
To find the employees with the highest salary within each education level, you can use the following SQL query:

```sql
WITH RankedSalaries AS (
    SELECT 
        e.name AS employee_name,
        s.salary,
        st.educational_level,
        ROW_NUMBER() OVER(PARTITION BY st.educational_level ORDER BY s.salary DESC) AS rank
    FROM employees e
    JOIN salary s ON e.ID_Usr = s.ID_Usr
    JOIN studies st ON e.ID_Usr = st.ID_Usr
)
SELECT employee_name, salary, educational_level
FROM RankedSalaries
WHERE rank = 1;
```

This query uses a Common Table Expression (CTE) to rank the employees based on their salary within each education level. The final SELECT statement then filters out only the employees with the highest salary within each education level.
--------------------------------------------------------------------------------
QUESTION: How many employees graduated from each universi

## Conclusion

The results show that the new prompt generated more accurate SQL queries, and provided better explanations and handled more complex questions more effectively.

The old prompt worked well for simple questions but sometimes produced incomplete or less accurate queries. For example, it did not correctly answer the question about schools producing the highest-paid employees and oversimplified the revenue question.

Overall, the new prompt performed better because it included clearer database structure information and examples. This demonstrates that providing detailed schema information can improve SQL generation and help analysts obtain more accurate results.
